# 02 - Feature engineering, model development and tracking
The same logic as `python -m src.train`, step by step, so every decision is visible. MLflow logs go to a separate experiment so the notebook never pollutes the main one.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

from src import config
from src.data import load_clean, split_xy
from src.features import add_features, build_preprocessor, get_feature_names
from src.train import CV_SCORING, candidate_models, make_pipeline, setup_mlflow

setup_mlflow(f"sqlite:///{ROOT / 'mlflow.db'}", "heart-disease-notebook")

## 1. Split first, then preprocess
A stratified 80/20 split is made **before** any fitting. Imputation, scaling and encoding live inside a `Pipeline`, so they are re-fitted on each CV training fold and cannot leak test information.

In [3]:
df = load_clean()
X, y = split_xy(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.RANDOM_STATE)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

(242, 13) (61, 13) 0.459 0.459


## 2. Feature engineering
* numeric (`age, trestbps, chol, thalach, oldpeak`) + engineered `hr_ratio`: median impute, standardise
* categorical (`cp, restecg, slope, thal, ca`): most-frequent impute, one-hot
* binary (`sex, fbs, exang`): most-frequent impute, passthrough

`hr_ratio = thalach / (220 - age)` is the share of the age-predicted maximum heart rate the patient reached. It normalises `thalach` for age, since older patients naturally peak lower.

In [4]:
prep = build_preprocessor().fit(X_train)
Xt = prep.transform(X_train)
print("transformed shape:", Xt.shape)
pd.DataFrame(np.asarray(Xt), columns=get_feature_names(prep)).head()

transformed shape: (242, 26)


,age,trestbps,chol,thalach,oldpeak,hr_ratio,cp_1.0,cp_2.0,cp_3.0,cp_4.0,...,thal_3.0,thal_6.0,thal_7.0,ca_0.0,ca_1.0,ca_2.0,ca_3.0,sex,fbs,exang
0,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.465919,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.262646,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.061212,0.059213,0.723605,0.399178,-0.891627,0.408349,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
3,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.265532,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.272924,0.514117,-0.167601,-1.190962,-0.713154,-1.197694,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [5]:
# Does the engineered feature carry signal beyond its parts?
tmp = add_features(df)
print(tmp[["thalach", "hr_ratio", config.TARGET]].corr()[config.TARGET].round(3))

thalach    -0.417
hr_ratio   -0.359
target      1.000
Name: target, dtype: float64


`hr_ratio` is *less* correlated with the target (-0.36) than raw `thalach` (-0.42), so a quick ablation is needed before claiming it helps: repeated 5x5 CV ROC-AUC for logistic regression with and without it.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=config.RANDOM_STATE)

def ablation_auc():
    model = make_pipeline(LogisticRegression(max_iter=2000, C=1))
    return cross_val_score(model, X_train, y_train, cv=rcv, scoring="roc_auc").mean()

auc_with = ablation_auc()
saved = list(config.ENGINEERED_FEATURES)
config.ENGINEERED_FEATURES.clear()           # temporarily drop hr_ratio
auc_without = ablation_auc()
config.ENGINEERED_FEATURES.extend(saved)     # restore
print(f"CV ROC-AUC with hr_ratio: {auc_with:.4f} | without: {auc_without:.4f}")

CV ROC-AUC with hr_ratio: 0.8928 | without: 0.8928


**Result:** no measurable gain for logistic regression - `hr_ratio` is almost a linear combination of `age` and `thalach`, which the model already has. It is kept in the pipeline as a documented domain feature (and can help tree models), but I do not claim it improves performance.

## 3. Cross-validated comparison with tuning
Stratified 5-fold CV, `GridSearchCV` optimising ROC-AUC. Three candidates of increasing flexibility: logistic regression (linear baseline), random forest, gradient boosting.

In [7]:
cv = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
from sklearn.model_selection import GridSearchCV

rows, best_models = [], {}
for name, (est, grid) in candidate_models().items():
    search = GridSearchCV(make_pipeline(est), grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_train, y_train)
    res = cross_validate(search.best_estimator_, X_train, y_train, cv=cv, scoring=CV_SCORING)
    row = {"model": name, "best_params": search.best_params_}
    for k in CV_SCORING:
        row[k] = f"{res[f'test_{k}'].mean():.3f} +/- {res[f'test_{k}'].std():.3f}"
    rows.append(row); best_models[name] = search.best_estimator_
pd.DataFrame(rows).set_index("model")

,best_params,accuracy,precision,recall,f1,roc_auc
model,,,,,,
logistic_regression,{'clf__C': 1},0.847 +/- 0.010,0.877 +/- 0.054,0.783 +/- 0.046,0.825 +/- 0.007,0.904 +/- 0.014
random_forest,"{'clf__max_depth': 3, 'clf__min_samples_leaf':...",0.826 +/- 0.034,0.860 +/- 0.073,0.755 +/- 0.095,0.797 +/- 0.044,0.896 +/- 0.025
gradient_boosting,"{'clf__learning_rate': 0.05, 'clf__max_depth':...",0.822 +/- 0.026,0.829 +/- 0.064,0.783 +/- 0.068,0.801 +/- 0.030,0.878 +/- 0.028


## 4. Held-out test evaluation

In [8]:
from src.train import evaluate_on_test
test_df = pd.DataFrame({n: evaluate_on_test(m, X_test, y_test) for n, m in best_models.items()}).T.round(3)
test_df

,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
logistic_regression,0.885,0.839,0.929,0.881,0.968
random_forest,0.852,0.828,0.857,0.842,0.949
gradient_boosting,0.902,0.867,0.929,0.897,0.959


In [9]:
from sklearn.metrics import RocCurveDisplay
fig, ax = plt.subplots(figsize=(6, 5))
for n, m in best_models.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, ax=ax, name=n)
ax.plot([0, 1], [0, 1], "k--", alpha=.4); ax.set_title("ROC - held-out test set")
plt.show()

## 5. Log this notebook run to MLflow

In [10]:
for name, model in best_models.items():
    with mlflow.start_run(run_name=f"nb-{name}"):
        mlflow.log_param("model_type", name)
        mlflow.log_metrics({k: float(v) for k, v in test_df.loc[name].items()})
print("Logged. Open the UI with: mlflow ui --backend-store-uri sqlite:///mlflow.db")

Logged. Open the UI with: mlflow ui --backend-store-uri sqlite:///mlflow.db


### Reading the results honestly
* Selection is made on **CV ROC-AUC**, never on the test set.
* The CV figures for the tuned model are slightly optimistic because the same folds picked the hyper-parameters; the held-out test set is the cleaner estimate.
* With only 303 rows the test split has ~61 patients, so a single misclassification moves accuracy by ~1.6 points. Differences of a few points between models are within noise; the CV standard deviations make this visible.
* The test AUC being higher than CV AUC for every model is a sign of an easy split, not of a better model.